# Experiment 3: DenseNet-201 + ViT-B/16 Feature-Level Fusion Multi-Task Baseline - BreakHis 200X

**Objective:**
Build, evaluate, and benchmark the first hybrid feature-level fusion multi-task model (DenseNet-201 CNN + ViT-B/16 Transformer) on BreakHis 200X breast cancer histopathology images under the **corrected evaluation protocol**.

---

### Research Context & Corrected Evaluation Protocol

1. **Flaw Discovered in Baseline 1 & 2:**
   - BreakHis contains only 82 patients across 8 subtypes.
   - A single 3-way split produced `support = 0` for rare subtypes (such as phyllodes tumor) in the held-out test partition.
2. **Corrected Evaluation Strategy:**
   - **Task A (Primary: Benign vs Malignant):** Evaluated on a single held-out stratified 3-way patient split (`split_task_a.csv`, 70/15/15).
   - **Task B (Subtypes: 8 Classes):** Evaluated via **patient-level stratified 5-fold cross-validation** (`folds_task_b.csv`).
   - **Nested Patient-Level Inner Validation:** In each outer fold, the training cohort is partitioned into inner-train (~80%) and inner-val (~20%) at the patient level for early stopping. The outer test fold remains strictly held out.
   - **Honest Rare-Class Reporting:** Subtypes with insufficient patient count (< 5 patients) are explicitly flagged rather than silently reported as 0.0 or omitted.
3. **Leakage-Free Apples-to-Apples Comparison:**
   - Baseline 1 (DenseNet-201) and Baseline 2 (ViT-B/16) are retrained and evaluated on the exact same 5-fold CV protocol alongside Baseline 3 Fusion.
4. **Pre-Registered Success Criterion:**
   $$\text{Fusion Subtype Macro-F1} \ge \max(\text{DenseNet Macro-F1}, \text{ViT Macro-F1}) + 1.0 \times \sigma_{\text{cross-fold}}$$

---

### Hybrid Architecture Overview

```
                        Input Image (224x224x3)
                                  |
                +-----------------+-----------------+
                |                                   |
                v                                   v
        DenseNet-201 (CNN)                  ViT-B/16 (Transformer)
        1920-d Pooled Feature               768-d [CLS] Feature
                |                                   |
                v                                   v
        Linear(1920 -> 384)                 Linear(768 -> 384)
        + BatchNorm1d + ReLU + Drop         + LayerNorm + ReLU + Drop
                |                                   |
                +-----------------+-----------------+
                                  |
                          Concatenation (768-d)
                                  |
                                  v
                        Fusion MLP (768 -> 384)
                        + ReLU + Dropout(0.3)
                                  |
                +-----------------+-----------------+
                |                                   |
                v                                   v
         Head A (Primary)                    Head B (Subtype)
         Linear(384 -> 2)                    Linear(384 -> 8)
```

In [ ]:
# ============================================================
# Environment Setup & Google Drive Persistence
# ============================================================
import os
import sys

# Mount Google Drive for persistent artifact and checkpoint storage
try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/output_base3'
    print("[OK] Google Drive mounted successfully.")
except Exception as e:
    SAVE_DIR = './output_base3'
    print(f"[INFO] Google Colab Drive not detected. Falling back to local directory: {SAVE_DIR}")

# Create output directory for Baseline 3 artifacts
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"[OK] Artifact directory ready: {SAVE_DIR}")

## Kaggle API Authentication & Secure Credential Handling

To download the private BreakHis dataset (`trexbytes/breakhislink`), authenticate Kaggle in one of the following ways:

1. **Option A (Google Colab Secrets - Recommended):**
   - Click the Key icon (Secrets) in the left panel of Colab.
   - Add `KAGGLE_API_TOKEN` with your Kaggle token (or `KAGGLE_USERNAME` and `KAGGLE_KEY`).
   - Enable notebook access.
2. **Option B (Direct Environment Variable in Cell 3):**
   - Set `os.environ["KAGGLE_API_TOKEN"] = "your_token"` in Cell 3.
3. **Option C (Interactive Secure Input):**
   - If not set, running Cell 3 will prompt you to enter your token securely via `getpass`.

> **Note:** Kaggle authentication is required to access private datasets.

In [ ]:
# ============================================================
# Kaggle API Authentication
# ============================================================
import os
import json
import getpass

# 1. Check for Colab Secrets (google.colab.userdata)
try:
    from google.colab import userdata
    try:
        token = userdata.get('KAGGLE_API_TOKEN')
        if token:
            os.environ['KAGGLE_API_TOKEN'] = str(token).strip()
            print("[OK] Kaggle API token loaded from Colab Secrets (KAGGLE_API_TOKEN).")
    except Exception:
        pass
    try:
        uname = userdata.get('KAGGLE_USERNAME')
        ukey = userdata.get('KAGGLE_KEY')
        if uname and ukey:
            os.environ['KAGGLE_USERNAME'] = str(uname).strip()
            os.environ['KAGGLE_KEY'] = str(ukey).strip()
            print("[OK] Kaggle credentials loaded from Colab Secrets (KAGGLE_USERNAME, KAGGLE_KEY).")
    except Exception:
        pass
except ImportError:
    pass

# 2. Check for environment variables or prompt if not present
if not os.environ.get('KAGGLE_API_TOKEN') and not (os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY')):
    kaggle_json_path = os.path.expanduser('~/.kaggle/kaggle.json')
    if os.path.exists(kaggle_json_path):
        print(f"[OK] Existing kaggle.json detected at {kaggle_json_path}.")
    else:
        # Prompt user securely in Colab
        print("[INFO] Kaggle credentials not found in Colab Secrets or environment.")
        user_token = getpass.getpass("Enter your Kaggle API Token: ").strip()
        if user_token:
            os.environ['KAGGLE_API_TOKEN'] = user_token
            print("[OK] Kaggle API Token set for current session.")

# 3. Create ~/.kaggle/kaggle.json for legacy Kaggle CLI compatibility if credentials exist
kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
if os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY'):
    with open(os.path.join(kaggle_dir, 'kaggle.json'), 'w') as f:
        json.dump({
            'username': os.environ['KAGGLE_USERNAME'],
            'key': os.environ['KAGGLE_KEY']
        }, f)
    os.chmod(os.path.join(kaggle_dir, 'kaggle.json'), 0o600)

if os.environ.get('KAGGLE_API_TOKEN'):
    print("[OK] Kaggle authentication configured via KAGGLE_API_TOKEN.")
elif os.environ.get('KAGGLE_USERNAME'):
    print("[OK] Kaggle authentication configured via KAGGLE_USERNAME/KEY.")
else:
    print("[WARNING] No Kaggle credentials detected. If the dataset is private, download may fail.")

In [ ]:
# ============================================================
# Dataset Ingestion via Kagglehub (Direct Streaming)
# ============================================================
import glob
import zipfile
import shutil

# Install / import kagglehub
try:
    import kagglehub
except ImportError:
    !pip install -q kagglehub
    import kagglehub

print("Downloading BreakHis dataset from Kaggle (trexbytes/breakhislink)...")
try:
    dataset_cache_path = kagglehub.dataset_download("trexbytes/breakhislink")
    print(f"[OK] Path to dataset files: {dataset_cache_path}")
except Exception as e:
    print(f"[ERROR] Kaggle download failed: {e}")
    print("\n[TROUBLESHOOTING]:")
    print("1. Ensure you ran Cell 3 and provided a valid Kaggle API Token.")
    print("2. You can manually set in a code cell: os.environ['KAGGLE_API_TOKEN'] = 'your_token'")
    print("3. Then re-run this cell.")
    raise e

# Check if downloaded directory contains zip files that need extraction to local disk
existing_pngs = glob.glob(os.path.join(dataset_cache_path, '**', '*.png'), recursive=True)
if len(existing_pngs) == 0:
    zip_files = glob.glob(os.path.join(dataset_cache_path, '**', '*.zip'), recursive=True)
    if zip_files:
        local_extract_dir = '/content/breakhis_data'
        os.makedirs(local_extract_dir, exist_ok=True)
        for zf in zip_files:
            print(f"[INFO] Extracting {os.path.basename(zf)} to local cache {local_extract_dir}...")
            with zipfile.ZipFile(zf, 'r') as z:
                z.extractall(local_extract_dir)
        dataset_cache_path = local_extract_dir
        existing_pngs = glob.glob(os.path.join(dataset_cache_path, '**', '*.png'), recursive=True)

print(f"[OK] Dataset ready: {len(existing_pngs)} PNG images found in {dataset_cache_path}")
assert len(existing_pngs) > 0, f"[ERROR] No PNG images found in {dataset_cache_path}!"

In [ ]:
# ============================================================
# Experiment Configuration (CONFIG)
# ============================================================

CONFIG = {
    'experiment_name': 'baseline_3_densenet_vit_fusion',
    'seed': 42,
    
    'data': {
        'dataset_path': dataset_cache_path,
        'magnification': '200X',           # Retain 200X magnification lock
        'input_size': 224,                  # Standard 224x224 input resolution
        'batch_size': 16,                  # Small batch size for Colab memory safety
        'num_workers': 0,                  # MANDATORY: 0 workers to prevent multiprocessing child-process crashes
        'pin_memory': True,
    },
    
    'augmentation': {
        'horizontal_flip': True,
        'vertical_flip': True,
        'rotation_degrees': 20,
        'color_jitter': True,
    },
    
    'model': {
        'cnn_backbone': 'densenet201',
        'cnn_emb_dim': 1920,
        'vit_backbone': 'vit_base_patch16_224',
        'vit_emb_dim': 768,
        'proj_dim': 384,                   # Projection dimension for both branches
        'fused_dim': 768,                  # 384 + 384 = 768
        'fusion_hidden_dim': 384,          # Fusion MLP intermediate dimension
        'primary_num_classes': 2,          # Benign vs Malignant
        'subtype_num_classes': 8,          # 8 histopathological subtypes
        'dropout': 0.3,
        'pretrained': True,
    },
    
    'training': {
        'k_folds': 5,                      # 5-fold patient-level cross-validation for Task B
        'phase1_epochs': 15,               # Epochs for frozen backbone training
        'phase2_epochs': 15,               # Epochs for partial unfreeze fine-tuning
        'unfreeze_vit_last_n_blocks': 2,    # Unfreeze the last 2 transformer blocks of ViT in Phase 2
        'lr_head': 1e-3,                   # Learning rate for projection layers & multi-task heads
        'lr_backbone': 1e-5,               # Discriminative lower LR for unfreezed backbone layers
        'weight_decay': 1e-4,
        'subtype_loss_weight': 1.0,        # lambda = 1.0 multi-task loss balance
        'label_smoothing': 0.05,
        'early_stopping_patience': 7,
        'mixed_precision': True,
    },
    
    'paths': {
        'save_dir': SAVE_DIR,
        'split_task_a_csv': os.path.join(SAVE_DIR, 'split_task_a.csv'),
        'folds_task_b_csv': os.path.join(SAVE_DIR, 'folds_task_b.csv'),
        'best_fusion_checkpoint': os.path.join(SAVE_DIR, 'best_fusion_model.pth'),
        'final_fusion_checkpoint': os.path.join(SAVE_DIR, 'final_fusion_model.pth'),
        'metrics_json': os.path.join(SAVE_DIR, 'test_metrics_base3.json'),
        'comparison_csv': os.path.join(SAVE_DIR, 'benchmark_comparison.csv'),
    },
    
    'success_criterion': {
        'description': "Fusion Subtype Macro-F1 >= Best Baseline Macro-F1 + 1.0 * cross_fold_std",
        'rule': 'mean_plus_1_std',
    },
    
    'smoke_test': False                     # Set True for rapid 1-batch dry run
}

print("[OK] CONFIG dictionary initialized.")
print(f"   Magnification lock: {CONFIG['data']['magnification']}")
print(f"   DataLoader workers: {CONFIG['data']['num_workers']}")
print(f"   Output Directory:   {CONFIG['paths']['save_dir']}")

In [ ]:
# ============================================================
# Imports, Reproducibility & Device Configuration
# ============================================================
import random
import time
import glob
import math
import copy
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import timm

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from sklearn.utils.class_weight import compute_class_weight

# --- Reproducibility Seed Everything ---
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(CONFIG['seed'])

# --- Device & Memory Verification ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[OK] Compute Device: {device}")

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    # Using .total_memory as per notebook guidelines (not .total_mem)
    total_mem_gb = props.total_memory / (1024 ** 3)
    print(f"   GPU Model:        {props.name}")
    print(f"   Total VRAM:       {total_mem_gb:.2f} GB")
    print(f"   CUDA Capability:  {props.major}.{props.minor}")
    print(f"   PyTorch Version:  {torch.__version__}")
else:
    print("   [WARNING] Running on CPU! Training will be significantly slower.")

## Mandatory Pre-Training Data Audit & Corrected Split Generation

### Audit Objectives:
1. Scan BreakHis dataset directory structure and locate all images matching `200X` magnification.
2. Extract exact patient / case IDs from filenames (e.g., `SOB_B_A-14-22549AB-200-001.png` $\to$ Patient `A_14-22549AB`).
3. Audit patient count per subtype:
   - Identify subtypes with $\ge 5$ patients (support full 5-fold CV).
   - Identify rare subtypes with $< 5$ patients (e.g. phyllodes tumor: 3 patients, adenosis: 4 patients) and flag them explicitly.
4. Generate canonical evaluation artifacts:
   - `split_task_a.csv`: Single 3-way patient-level split (70% train, 15% val, 15% test) for Task A (Primary classification).
   - `folds_task_b.csv`: 5-fold patient-level stratified fold assignments for Task B (Subtype classification).
5. Assert **zero patient overlap** across splits and folds.

In [ ]:
# ============================================================
# Directory Structure Scan & Filename Parsing
# ============================================================
print("=" * 60)
print("PRE-TRAINING DATA AUDIT: BREAKHIS 200X")
print("=" * 60)

raw_data_dir = CONFIG['data']['dataset_path']
target_mag = CONFIG['data']['magnification']

# Subtype folders and codes
SUBTYPE_FOLDERS = {
    'adenosis', 'fibroadenoma', 'phyllodes_tumor', 'tubular_adenoma',
    'ductal_carcinoma', 'lobular_carcinoma', 'mucinous_carcinoma', 'papillary_carcinoma'
}

SUBTYPE_CODE_MAP = {
    'A': 'adenosis', 'F': 'fibroadenoma', 'PT': 'phyllodes_tumor', 'TA': 'tubular_adenoma',
    'DC': 'ductal_carcinoma', 'LC': 'lobular_carcinoma', 'MC': 'mucinous_carcinoma', 'PC': 'papillary_carcinoma'
}

PRIMARY_FROM_SUBTYPE = {
    'adenosis': 'benign', 'fibroadenoma': 'benign', 'phyllodes_tumor': 'benign', 'tubular_adenoma': 'benign',
    'ductal_carcinoma': 'malignant', 'lobular_carcinoma': 'malignant', 'mucinous_carcinoma': 'malignant', 'papillary_carcinoma': 'malignant'
}

# Scan for all PNG images
all_image_paths = []
for root, _, files in os.walk(raw_data_dir):
    for f in files:
        if f.lower().endswith('.png'):
            all_image_paths.append(os.path.join(root, f))

print(f"[OK] Total PNG images scanned across all magnifications: {len(all_image_paths)}")

# Filter for target magnification (200X)
mag_tag = f"-{target_mag.lower()}-"
mag_tag_alt = f"/{target_mag.lower()}/"
mag_tag_alt2 = f"\\{target_mag.lower()}\\"

mag_images = [
    p for p in all_image_paths
    if mag_tag in p.lower() or mag_tag_alt in p.lower() or mag_tag_alt2 in p.lower() or f"-{target_mag}-" in p
]

print(f"[OK] Images matching magnification lock ({target_mag}): {len(mag_images)}")
assert len(mag_images) > 0, f"[ERROR] No {target_mag} images found in {raw_data_dir}!"

# Parse each image metadata
records = []
parse_errors = []

for img_path in mag_images:
    norm_path = img_path.replace('\\', '/')
    fname = os.path.basename(img_path)
    fname_no_ext = os.path.splitext(fname)[0]
    
    # 1. Determine subtype from directory path or filename
    subtype = None
    path_lower = norm_path.lower()
    for st in SUBTYPE_FOLDERS:
        if f"/{st}/" in path_lower or f"_{st}_" in path_lower:
            subtype = st
            break
            
    if subtype is None:
        prefix = fname_no_ext.split('-')[0]
        parts = prefix.split('_')
        if len(parts) >= 3:
            code_str = '_'.join(parts[2:])
            subtype = SUBTYPE_CODE_MAP.get(code_str)
            
    if subtype is None:
        parse_errors.append(f"Subtype undetermined: {fname}")
        continue
        
    primary = PRIMARY_FROM_SUBTYPE[subtype]
    
    # 2. Extract patient/case ID from filename (SOB_B_A-14-22549AB-200-001 -> A_14-22549AB)
    parts = fname_no_ext.split('-')
    if len(parts) >= 4:
        case_id = '-'.join(parts[1:-2])
        subtype_prefix = parts[0].split('_')[-1]
        patient_id = f"{subtype_prefix}_{case_id}"
    else:
        patient_id = fname_no_ext
        
    records.append({
        'filepath': img_path,
        'filename': fname,
        'patient_id': patient_id,
        'magnification': target_mag,
        'primary_label': primary,
        'subtype_label': subtype
    })

audit_df = pd.DataFrame(records)
print(f"[OK] Successfully parsed {len(audit_df)} valid image records.")
if parse_errors:
    print(f"[WARNING] Parse errors encountered on {len(parse_errors)} files.")

# Print class and patient distributions
print("\n" + "=" * 60)
print("CLASS & PATIENT DISTRIBUTION REPORT")
print("=" * 60)

print("\n--- Task A: Primary Classification (Benign vs Malignant) ---")
for label, count in audit_df['primary_label'].value_counts().items():
    pct = 100.0 * count / len(audit_df)
    print(f"  {label:12s}: {count:5d} images ({pct:.1f}%)")

print("\n--- Task B: Subtype Classification (8 classes) ---")
for label, count in audit_df['subtype_label'].value_counts().items():
    pct = 100.0 * count / len(audit_df)
    print(f"  {label:25s}: {count:5d} images ({pct:.1f}%)")

print("\n--- Unique Patient Counts per Subtype ---")
n_patients = audit_df['patient_id'].nunique()
print(f"  Total unique patients/cases: {n_patients}")

patient_subtype_counts = audit_df.groupby('patient_id')['subtype_label'].first().value_counts()
for label, count in patient_subtype_counts.items():
    flag = " [OK: >= 5 patients]" if count >= 5 else " [FLAG: < 5 patients - rare class]"
    print(f"  {label:25s}: {count:3d} patients{flag}")

In [ ]:
# ============================================================
# Corrected Evaluation Artifacts: split_task_a.csv & folds_task_b.csv
# ============================================================
print("=" * 60)
print("GENERATING CORRECTED EVALUATION ARTIFACTS")
print("=" * 60)

# 1. Unique patient table with metadata
patient_summary = audit_df.groupby('patient_id').agg({
    'subtype_label': 'first',
    'primary_label': 'first',
    'filepath': 'count'
}).rename(columns={'filepath': 'image_count'}).reset_index()

# ------------------------------------------------------------
# Artifact 1: split_task_a.csv (Task A Single 3-Way Patient Split: 70% Train, 15% Val, 15% Test)
# ------------------------------------------------------------
def generate_task_a_split(patient_df, seed=42):
    random.seed(seed)
    np.random.seed(seed)
    
    split_records = []
    for subtype, grp in patient_df.groupby('subtype_label'):
        pids = list(grp['patient_id'].values)
        random.shuffle(pids)
        n = len(pids)
        
        if n >= 6:
            n_test = max(1, int(round(n * 0.15)))
            n_val = max(1, int(round(n * 0.15)))
        elif n >= 4:
            n_test = 1
            n_val = 1
        elif n >= 2:
            n_test = 1
            n_val = 0
        else:
            n_test = 0
            n_val = 0
            
        test_p = set(pids[:n_test])
        val_p = set(pids[n_test:n_test + n_val])
        train_p = set(pids[n_test + n_val:])
        
        for pid in pids:
            sp = 'test' if pid in test_p else ('val' if pid in val_p else 'train')
            split_records.append({'patient_id': pid, 'split_task_a': sp})
            
    return pd.DataFrame(split_records)

task_a_patient_splits = generate_task_a_split(patient_summary, seed=CONFIG['seed'])
split_task_a_df = audit_df.merge(task_a_patient_splits, on='patient_id')
split_task_a_df = split_task_a_df.rename(columns={'split_task_a': 'split'})

# Save split_task_a.csv
split_task_a_path = CONFIG['paths']['split_task_a_csv']
split_task_a_df[['filepath', 'patient_id', 'magnification', 'primary_label', 'subtype_label', 'split']].to_csv(
    split_task_a_path, index=False
)
print(f"[OK] Task A split saved to: {split_task_a_path}")

# Verify zero patient overlap in Task A
p_train = set(split_task_a_df[split_task_a_df['split'] == 'train']['patient_id'])
p_val = set(split_task_a_df[split_task_a_df['split'] == 'val']['patient_id'])
p_test = set(split_task_a_df[split_task_a_df['split'] == 'test']['patient_id'])
assert len(p_train & p_val) == 0, "[ERROR] Task A leak: train & val"
assert len(p_train & p_test) == 0, "[ERROR] Task A leak: train & test"
assert len(p_val & p_test) == 0, "[ERROR] Task A leak: val & test"
print("   [OK] Zero patient leakage in split_task_a.csv verified.")

# ------------------------------------------------------------
# Artifact 2: folds_task_b.csv (Task B Patient-Level Stratified 5-Fold Assignments)
# ------------------------------------------------------------
def generate_task_b_folds(patient_df, seed=42, n_splits=5):
    random.seed(seed)
    np.random.seed(seed)
    
    fold_records = []
    for subtype, grp in patient_df.groupby('subtype_label'):
        pids = list(grp['patient_id'].values)
        random.shuffle(pids)
        for i, pid in enumerate(pids):
            assigned_fold = i % n_splits
            fold_records.append({'patient_id': pid, 'fold': assigned_fold})
            
    return pd.DataFrame(fold_records)

task_b_patient_folds = generate_task_b_folds(patient_summary, seed=CONFIG['seed'], n_splits=CONFIG['training']['k_folds'])
folds_task_b_df = audit_df.merge(task_b_patient_folds, on='patient_id')

# Save folds_task_b.csv
folds_task_b_path = CONFIG['paths']['folds_task_b_csv']
folds_task_b_df[['filepath', 'patient_id', 'magnification', 'primary_label', 'subtype_label', 'fold']].to_csv(
    folds_task_b_path, index=False
)
print(f"[OK] Task B 5-fold assignments saved to: {folds_task_b_path}")

# Verify 5-fold patient partitioning
print("\n--- Task B Subtype Fold Distribution (Patient Counts) ---")
patient_fold_cross = folds_task_b_df.groupby(['subtype_label', 'fold'])['patient_id'].nunique().unstack(fill_value=0)
print(patient_fold_cross)

# Identify and document rare classes
rare_subtypes = [st for st, count in patient_subtype_counts.items() if count < CONFIG['training']['k_folds']]
print(f"\n[INFO] Rare subtypes with insufficient patient support (< {CONFIG['training']['k_folds']} patients): {rare_subtypes}")
print("   These classes will be explicitly tracked and flagged across test folds.")

## Data Pipeline & Precomputed Feature Caching Engine

### Data Pipeline Architecture:
- Standardized image input size: **224x224x3**
- Train augmentations: RandomHorizontalFlip, RandomVerticalFlip, RandomRotation(20°), ColorJitter
- Normalization: ImageNet statistics `[0.485, 0.456, 0.406]` / `[0.229, 0.224, 0.225]`
- PyTorch `DataLoader` with **`num_workers = 0`** (mandatory for Colab child-process stability)

### Phase 1 Feature Caching Lever:
- Because backbone weights are frozen in Phase 1, feature representations (1920-d DenseNet, 768-d ViT) can be **precomputed once** for the entire 200X dataset.
- Training projection layers and fusion MLPs directly from cached embeddings accelerates 5-fold CV by ~50x, completely eliminating GPU OOM and redundant forward passes.

In [ ]:
# ============================================================
# PyTorch Dataset Classes & Transforms
# ============================================================

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
input_size = CONFIG['data']['input_size']

# Label mappings
PRIMARY_LABEL_MAP = {'benign': 0, 'malignant': 1}
PRIMARY_IDX_TO_LABEL = {v: k for k, v in PRIMARY_LABEL_MAP.items()}

SUBTYPE_LABEL_MAP = {
    'adenosis': 0, 'fibroadenoma': 1, 'phyllodes_tumor': 2, 'tubular_adenoma': 3,
    'ductal_carcinoma': 4, 'lobular_carcinoma': 5, 'mucinous_carcinoma': 6,
    'papillary_carcinoma': 7
}
SUBTYPE_IDX_TO_LABEL = {v: k for k, v in SUBTYPE_LABEL_MAP.items()}

# Transforms
aug_cfg = CONFIG['augmentation']
train_tfm_list = [transforms.Resize((input_size, input_size))]
if aug_cfg['horizontal_flip']:
    train_tfm_list.append(transforms.RandomHorizontalFlip(p=0.5))
if aug_cfg['vertical_flip']:
    train_tfm_list.append(transforms.RandomVerticalFlip(p=0.5))
if aug_cfg['rotation_degrees'] > 0:
    train_tfm_list.append(transforms.RandomRotation(aug_cfg['rotation_degrees']))
if aug_cfg['color_jitter']:
    train_tfm_list.append(
        transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05, hue=0.02)
    )
train_tfm_list += [transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)]
train_transform = transforms.Compose(train_tfm_list)

val_transform = transforms.Compose([
    transforms.Resize((input_size, input_size)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# --- Image Dataset ---
class BreakHisDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['filepath']).convert('RGB')
        if self.transform:
            image = self.transform(image)
        primary = PRIMARY_LABEL_MAP[row['primary_label']]
        subtype = SUBTYPE_LABEL_MAP[row['subtype_label']]
        return image, primary, subtype

# --- Cached Feature Dataset for Fast Phase 1 Training ---
class CachedFeatureDataset(Dataset):
    def __init__(self, cnn_features, vit_features, primary_labels, subtype_labels):
        self.cnn_features = torch.as_tensor(cnn_features, dtype=torch.float32)
        self.vit_features = torch.as_tensor(vit_features, dtype=torch.float32)
        self.primary_labels = torch.as_tensor(primary_labels, dtype=torch.long)
        self.subtype_labels = torch.as_tensor(subtype_labels, dtype=torch.long)

    def __len__(self):
        return len(self.primary_labels)

    def __getitem__(self, idx):
        return (
            self.cnn_features[idx],
            self.vit_features[idx],
            self.primary_labels[idx],
            self.subtype_labels[idx]
        )

# --- Class Weights Computation ---
train_pri = split_task_a_df[split_task_a_df['split'] == 'train']['primary_label'].map(PRIMARY_LABEL_MAP).values
train_sub = split_task_a_df[split_task_a_df['split'] == 'train']['subtype_label'].map(SUBTYPE_LABEL_MAP).values

pri_classes = np.array(sorted(PRIMARY_LABEL_MAP.values()))
pri_weights = compute_class_weight('balanced', classes=pri_classes, y=train_pri)
pri_weights_tensor = torch.tensor(pri_weights, dtype=torch.float32).to(device)

sub_classes = np.array(sorted(SUBTYPE_LABEL_MAP.values()))
sub_weights = compute_class_weight('balanced', classes=sub_classes, y=train_sub)
sub_weights_tensor = torch.tensor(sub_weights, dtype=torch.float32).to(device)

print("[OK] Dataset classes, transforms, and class weights initialized.")
print(f"   Primary weights: {pri_weights_tensor.cpu().numpy().round(3)}")
print(f"   Subtype weights: {sub_weights_tensor.cpu().numpy().round(3)}")

## Model Architectures (DenseNet-201, ViT-B/16, Feature Fusion Hybrid)

| Model | Backbone | Feature Dim | Projection | Fusion MLP | Heads |
|---|---|---|---|---|---|
| **Baseline 1** | DenseNet-201 | 1920-d | None | None | 1920 $\to$ 2, 1920 $\to$ 8 |
| **Baseline 2** | ViT-B/16 | 768-d | None | None | 768 $\to$ 2, 768 $\to$ 8 |
| **Baseline 3 (Hybrid)** | DenseNet-201 + ViT-B/16 | 1920-d + 768-d | 1920 $\to$ 384, 768 $\to$ 384 | 768 $\to$ 384 | 384 $\to$ 2, 384 $\to$ 8 |

---

### Key Architectural Specifications:
- **DenseNet Projection:** `Linear(1920, 384) -> BatchNorm1d(384) -> ReLU() -> Dropout(0.3)`
- **ViT Projection:** `Linear(768, 384) -> LayerNorm(384) -> ReLU() -> Dropout(0.3)`
- **Concatenation:** $[384, 384] \to 768$-d fused feature
- **Fusion MLP:** `Linear(768, 384) -> ReLU() -> Dropout(0.3)`
- **Multi-Task Heads:** Linear(384, 2) primary head, Linear(384, 8) subtype head.

In [ ]:
# ============================================================
# Model Architectures: DenseNet-201, ViT-B/16, Feature Fusion Hybrid
# ============================================================

# ------------------------------------------------------------
# 1. Standalone DenseNet-201 Multi-Task Model
# ------------------------------------------------------------
class DenseNet201MultiTask(nn.Module):
    def __init__(self, config):
        super().__init__()
        emb_dim = config['model']['cnn_emb_dim']
        dropout = config['model']['dropout']
        n_pri = config['model']['primary_num_classes']
        n_sub = config['model']['subtype_num_classes']
        pretrained = config['model']['pretrained']

        weights = models.DenseNet201_Weights.IMAGENET1K_V1 if pretrained else None
        densenet = models.densenet201(weights=weights)
        self.features = densenet.features
        self.relu = nn.ReLU(inplace=True)
        self.pool = nn.AdaptiveAvgPool2d((1, 1))

        self.head_primary = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(emb_dim, n_pri)
        )
        self.head_subtype = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(emb_dim, n_sub)
        )

    def extract_features(self, x):
        feat = self.features(x)
        feat = self.relu(feat)
        feat = self.pool(feat)
        return torch.flatten(feat, 1)

    def forward(self, x):
        emb = self.extract_features(x)
        return self.head_primary(emb), self.head_subtype(emb), emb

    def freeze_backbone(self):
        for param in self.features.parameters():
            param.requires_grad = False

    def unfreeze_denseblock4(self):
        # Unfreeze denseblock4 and norm5
        for name, param in self.features.named_parameters():
            if 'denseblock4' in name or 'norm5' in name:
                param.requires_grad = True

# ------------------------------------------------------------
# 2. Standalone ViT-B/16 Multi-Task Model
# ------------------------------------------------------------
class ViTB16MultiTask(nn.Module):
    def __init__(self, config):
        super().__init__()
        emb_dim = config['model']['vit_emb_dim']
        dropout = config['model']['dropout']
        n_pri = config['model']['primary_num_classes']
        n_sub = config['model']['subtype_num_classes']
        pretrained = config['model']['pretrained']

        self.backbone = timm.create_model(
            config['model']['vit_backbone'],
            pretrained=pretrained,
            num_classes=0
        )

        self.head_primary = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(emb_dim, n_pri)
        )
        self.head_subtype = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(emb_dim, n_sub)
        )

    def extract_features(self, x):
        return self.backbone(x)

    def forward(self, x):
        emb = self.extract_features(x)
        return self.head_primary(emb), self.head_subtype(emb), emb

    def freeze_backbone(self):
        for param in self.backbone.parameters():
            param.requires_grad = False

    def unfreeze_last_blocks(self, n=2):
        if hasattr(self.backbone, 'norm') and self.backbone.norm is not None:
            for param in self.backbone.norm.parameters():
                param.requires_grad = True
        if hasattr(self.backbone, 'blocks'):
            for block in self.backbone.blocks[-n:]:
                for param in block.parameters():
                    param.requires_grad = True

# ------------------------------------------------------------
# 3. DenseNet-201 + ViT-B/16 Feature-Level Fusion Multi-Task Model
# ------------------------------------------------------------
class FeatureFusionMultiTaskModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        cnn_dim = config['model']['cnn_emb_dim']        # 1920
        vit_dim = config['model']['vit_emb_dim']        # 768
        proj_dim = config['model']['proj_dim']          # 384
        mlp_hidden = config['model']['fusion_hidden_dim'] # 384
        dropout = config['model']['dropout']
        n_pri = config['model']['primary_num_classes']
        n_sub = config['model']['subtype_num_classes']
        pretrained = config['model']['pretrained']

        # Backbones
        densenet = models.densenet201(
            weights=models.DenseNet201_Weights.IMAGENET1K_V1 if pretrained else None
        )
        self.cnn_features = densenet.features
        self.cnn_relu = nn.ReLU(inplace=True)
        self.cnn_pool = nn.AdaptiveAvgPool2d((1, 1))

        self.vit_backbone = timm.create_model(
            config['model']['vit_backbone'],
            pretrained=pretrained,
            num_classes=0
        )

        # Projections
        self.cnn_proj = nn.Sequential(
            nn.Linear(cnn_dim, proj_dim),
            nn.BatchNorm1d(proj_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout)
        )
        self.vit_proj = nn.Sequential(
            nn.Linear(vit_dim, proj_dim),
            nn.LayerNorm(proj_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout)
        )

        # Fusion MLP
        self.fused_dim = proj_dim * 2  # 384 + 384 = 768
        self.fusion_mlp = nn.Sequential(
            nn.Linear(self.fused_dim, mlp_hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout)
        )

        # Multi-Task Heads
        self.head_primary = nn.Linear(mlp_hidden, n_pri)
        self.head_subtype = nn.Linear(mlp_hidden, n_sub)

    def extract_cnn(self, x):
        feat = self.cnn_features(x)
        feat = self.cnn_relu(feat)
        feat = self.cnn_pool(feat)
        return torch.flatten(feat, 1)

    def extract_vit(self, x):
        return self.vit_backbone(x)

    def forward_from_embeddings(self, cnn_emb, vit_emb):
        p_cnn = self.cnn_proj(cnn_emb)
        p_vit = self.vit_proj(vit_emb)
        fused = torch.cat([p_cnn, p_vit], dim=1)
        hidden = self.fusion_mlp(fused)
        return self.head_primary(hidden), self.head_subtype(hidden), hidden

    def forward(self, x):
        cnn_emb = self.extract_cnn(x)
        vit_emb = self.extract_vit(x)
        return self.forward_from_embeddings(cnn_emb, vit_emb)

    def freeze_backbones(self):
        for param in self.cnn_features.parameters():
            param.requires_grad = False
        for param in self.vit_backbone.parameters():
            param.requires_grad = False

    def unfreeze_partial_backbones(self, vit_blocks=2):
        # Unfreeze DenseNet block4 & norm5
        for name, param in self.cnn_features.named_parameters():
            if 'denseblock4' in name or 'norm5' in name:
                param.requires_grad = True
        # Unfreeze ViT last n blocks & norm
        if hasattr(self.vit_backbone, 'norm') and self.vit_backbone.norm is not None:
            for param in self.vit_backbone.norm.parameters():
                param.requires_grad = True
        if hasattr(self.vit_backbone, 'blocks'):
            for block in self.vit_backbone.blocks[-vit_blocks:]:
                for param in block.parameters():
                    param.requires_grad = True

# Verification
print("Verifying FeatureFusionMultiTaskModel construction...")
test_fusion_model = FeatureFusionMultiTaskModel(CONFIG).to(device)
test_fusion_model.eval()

with torch.no_grad():
    dummy_cnn = torch.randn(2, 1920).to(device)
    dummy_vit = torch.randn(2, 768).to(device)
    lp, ls, emb = test_fusion_model.forward_from_embeddings(dummy_cnn, dummy_vit)
    assert lp.shape == (2, 2), f"Invalid primary logits shape: {lp.shape}"
    assert ls.shape == (2, 8), f"Invalid subtype logits shape: {ls.shape}"
    assert emb.shape == (2, 384), f"Invalid embedding shape: {emb.shape}"

print("[OK] FeatureFusionMultiTaskModel shapes verified: Proj=384, Fused=768, Output=2/8")
del test_fusion_model, dummy_cnn, dummy_vit, lp, ls, emb
torch.cuda.empty_cache()

## Training Utilities & Metrics Computation

### Multi-Task Loss Formulation:
$$\mathcal{L}_{\text{total}} = \mathcal{L}_{\text{primary}} + \lambda \times \mathcal{L}_{\text{subtype}}$$
- Primary Loss: Weighted CrossEntropy with Label Smoothing ($0.05$)
- Subtype Loss: Weighted CrossEntropy with Label Smoothing ($0.05$)
- $\lambda = 1.0$ (matching Baseline 1 and 2 specifications)

### Early Stopping & Nested Inner Validation:
- Early stopping monitors `val_subtype_macro_f1` with patience $= 7$.
- In each fold $k$, model selection is executed strictly on the **inner validation set** derived from patients in folds $\neq k$. The outer test fold is evaluated exactly once on the selected best checkpoint.

In [ ]:
# ============================================================
# Training Utilities, Early Stopping & Evaluation Loop
# ============================================================

class EarlyStopping:
    def __init__(self, patience=7, mode='max', min_delta=0.0):
        self.patience = patience
        self.mode = mode
        self.min_delta = min_delta
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.best_state = None

    def __call__(self, val_score, model):
        score = val_score if self.mode == 'max' else -val_score
        if self.best_score is None:
            self.best_score = score
            self.best_state = copy.deepcopy(model.state_dict())
            return True
        elif score < self.best_score + self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
            return False
        else:
            self.best_score = score
            self.best_state = copy.deepcopy(model.state_dict())
            self.counter = 0
            return True

def compute_metrics(true_p, pred_p, true_s, pred_s):
    pri_acc = accuracy_score(true_p, pred_p)
    pri_f1 = f1_score(true_p, pred_p, average='macro', zero_division=0)
    sub_acc = accuracy_score(true_s, pred_s)
    sub_prec = precision_score(true_s, pred_s, average='macro', zero_division=0)
    sub_rec = recall_score(true_s, pred_s, average='macro', zero_division=0)
    sub_f1 = f1_score(true_s, pred_s, average='macro', zero_division=0)
    
    pc_f1 = f1_score(true_s, pred_s, average=None, zero_division=0)
    
    return {
        'primary_acc': pri_acc, 'primary_macro_f1': pri_f1,
        'subtype_acc': sub_acc, 'subtype_macro_prec': sub_prec,
        'subtype_macro_rec': sub_rec, 'subtype_macro_f1': sub_f1,
        'subtype_per_class_f1': pc_f1
    }

def train_cached_epoch(model, dataloader, optimizer, crit_p, crit_s, scaler, subtype_weight=1.0):
    model.train()
    total_loss, n = 0.0, 0
    preds_p, targs_p, preds_s, targs_s = [], [], [], []

    for cnn_emb, vit_emb, yp, ys in dataloader:
        cnn_emb = cnn_emb.to(device, non_blocking=True)
        vit_emb = vit_emb.to(device, non_blocking=True)
        yp = yp.to(device, non_blocking=True)
        ys = ys.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=CONFIG['training']['mixed_precision']):
            logits_p, logits_s, _ = model.forward_from_embeddings(cnn_emb, vit_emb)
            loss_p = crit_p(logits_p, yp)
            loss_s = crit_s(logits_s, ys)
            loss = loss_p + subtype_weight * loss_s

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        bs = yp.size(0)
        total_loss += loss.item() * bs
        n += bs

        preds_p.append(logits_p.argmax(1).cpu())
        targs_p.append(yp.cpu())
        preds_s.append(logits_s.argmax(1).cpu())
        targs_s.append(ys.cpu())

    p_p = torch.cat(preds_p).numpy(); t_p = torch.cat(targs_p).numpy()
    p_s = torch.cat(preds_s).numpy(); t_s = torch.cat(targs_s).numpy()
    metrics = compute_metrics(t_p, p_p, t_s, p_s)
    metrics['loss'] = total_loss / max(n, 1)
    return metrics

def eval_cached_epoch(model, dataloader, crit_p, crit_s, subtype_weight=1.0):
    model.eval()
    total_loss, n = 0.0, 0
    preds_p, targs_p, preds_s, targs_s = [], [], [], []

    with torch.no_grad():
        for cnn_emb, vit_emb, yp, ys in dataloader:
            cnn_emb = cnn_emb.to(device, non_blocking=True)
            vit_emb = vit_emb.to(device, non_blocking=True)
            yp = yp.to(device, non_blocking=True)
            ys = ys.to(device, non_blocking=True)

            with torch.amp.autocast('cuda', enabled=CONFIG['training']['mixed_precision']):
                logits_p, logits_s, _ = model.forward_from_embeddings(cnn_emb, vit_emb)
                loss_p = crit_p(logits_p, yp)
                loss_s = crit_s(logits_s, ys)
                loss = loss_p + subtype_weight * loss_s

            bs = yp.size(0)
            total_loss += loss.item() * bs
            n += bs

            preds_p.append(logits_p.argmax(1).cpu())
            targs_p.append(yp.cpu())
            preds_s.append(logits_s.argmax(1).cpu())
            targs_s.append(ys.cpu())

    p_p = torch.cat(preds_p).numpy(); t_p = torch.cat(targs_p).numpy()
    p_s = torch.cat(preds_s).numpy(); t_s = torch.cat(targs_s).numpy()
    metrics = compute_metrics(t_p, p_p, t_s, p_s)
    metrics['loss'] = total_loss / max(n, 1)
    return metrics, (t_p, p_p, t_s, p_s)

print("[OK] Training and evaluation utilities ready.")

## Feature Embedding Precomputation for Phase 1 Caching

We extract and cache the 1920-d DenseNet-201 and 768-d ViT-B/16 feature representations for all BreakHis 200X images. This enables rapid, memory-safe, and leak-free 5-fold cross-validation.

In [ ]:
# ============================================================
# Feature Embedding Precomputation (DenseNet-201 & ViT-B/16)
# ============================================================
print("=" * 60)
print("PRECOMPUTING BACKBONE FEATURE EMBEDDINGS")
print("=" * 60)

# Full dataset loader (evaluation mode, no augmentation)
full_eval_dataset = BreakHisDataset(folds_task_b_df, transform=val_transform)
full_eval_loader = DataLoader(
    full_eval_dataset, batch_size=CONFIG['data']['batch_size'], shuffle=False,
    num_workers=CONFIG['data']['num_workers'], pin_memory=CONFIG['data']['pin_memory']
)

# 1. DenseNet-201 Feature Extractor
print("\nExtracting DenseNet-201 embeddings (1920-d)...")
densenet_extractor = DenseNet201MultiTask(CONFIG).to(device)
densenet_extractor.eval()
cnn_embeddings = []

with torch.no_grad():
    for imgs, _, _ in tqdm(full_eval_loader, desc="DenseNet-201 Features"):
        imgs = imgs.to(device, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=CONFIG['training']['mixed_precision']):
            embs = densenet_extractor.extract_features(imgs)
        cnn_embeddings.append(embs.cpu())

all_cnn_feats = torch.cat(cnn_embeddings, dim=0).numpy()
print(f"[OK] DenseNet-201 features extracted: {all_cnn_feats.shape}")
del densenet_extractor, cnn_embeddings
torch.cuda.empty_cache()

# 2. ViT-B/16 Feature Extractor
print("\nExtracting ViT-B/16 embeddings (768-d)...")
vit_extractor = ViTB16MultiTask(CONFIG).to(device)
vit_extractor.eval()
vit_embeddings = []

with torch.no_grad():
    for imgs, _, _ in tqdm(full_eval_loader, desc="ViT-B/16 Features"):
        imgs = imgs.to(device, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=CONFIG['training']['mixed_precision']):
            embs = vit_extractor.extract_features(imgs)
        vit_embeddings.append(embs.cpu())

all_vit_feats = torch.cat(vit_embeddings, dim=0).numpy()
print(f"[OK] ViT-B/16 features extracted: {all_vit_feats.shape}")
del vit_extractor, vit_embeddings
torch.cuda.empty_cache()

# Target arrays
all_primary_targets = folds_task_b_df['primary_label'].map(PRIMARY_LABEL_MAP).values
all_subtype_targets = folds_task_b_df['subtype_label'].map(SUBTYPE_LABEL_MAP).values
all_patient_ids = folds_task_b_df['patient_id'].values
all_folds = folds_task_b_df['fold'].values

print(f"[OK] Feature precomputation complete: {len(all_patient_ids)} total image representations cached.")

## Task B: Baseline 1 (DenseNet-201) Retraining & 5-Fold Cross-Validation

Retrain DenseNet-201 multi-task heads under the corrected 5-fold CV protocol with nested patient-level inner validation for model selection.

In [ ]:
# ============================================================
# Baseline 1 (DenseNet-201) 5-Fold Cross-Validation
# ============================================================
print("=" * 60)
print("BASELINE 1 (DenseNet-201): 5-FOLD CROSS-VALIDATION")
print("=" * 60)

# Linear head model on top of cached DenseNet features
class DenseNetHeadModel(nn.Module):
    def __init__(self, cnn_dim=1920, n_pri=2, n_sub=8, dropout=0.3):
        super().__init__()
        self.head_primary = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(cnn_dim, n_pri)
        )
        self.head_subtype = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(cnn_dim, n_sub)
        )
    def forward_from_embeddings(self, cnn_emb, vit_emb=None):
        return self.head_primary(cnn_emb), self.head_subtype(cnn_emb), cnn_emb

b1_fold_results = []
b1_per_class_f1_list = []

for fold in range(CONFIG['training']['k_folds']):
    print(f"\n>>> Fold {fold + 1}/{CONFIG['training']['k_folds']}")
    
    # Outer split
    outer_test_mask = (all_folds == fold)
    outer_train_mask = (all_folds != fold)
    
    # Nested patient-level inner split
    outer_train_df = folds_task_b_df[outer_train_mask].copy()
    inner_val_pids = []
    for st, grp in outer_train_df.groupby('subtype_label'):
        pids = list(grp['patient_id'].unique())
        random.seed(CONFIG['seed'] + fold * 10)
        random.shuffle(pids)
        n = len(pids)
        n_val = max(1, int(round(n * 0.2))) if n >= 4 else (1 if n >= 2 else 0)
        inner_val_pids.extend(pids[:n_val])
        
    inner_val_mask = outer_train_mask & folds_task_b_df['patient_id'].isin(inner_val_pids).values
    inner_train_mask = outer_train_mask & (~folds_task_b_df['patient_id'].isin(inner_val_pids).values)
    
    # DataLoaders (with drop_last=True for training to avoid batch size 1 BatchNorm crashes)
    train_ds = CachedFeatureDataset(
        all_cnn_feats[inner_train_mask], all_vit_feats[inner_train_mask],
        all_primary_targets[inner_train_mask], all_subtype_targets[inner_train_mask]
    )
    val_ds = CachedFeatureDataset(
        all_cnn_feats[inner_val_mask], all_vit_feats[inner_val_mask],
        all_primary_targets[inner_val_mask], all_subtype_targets[inner_val_mask]
    )
    test_ds = CachedFeatureDataset(
        all_cnn_feats[outer_test_mask], all_vit_feats[outer_test_mask],
        all_primary_targets[outer_test_mask], all_subtype_targets[outer_test_mask]
    )
    
    train_loader = DataLoader(train_ds, batch_size=CONFIG['data']['batch_size'], shuffle=True, num_workers=0, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=CONFIG['data']['batch_size'], shuffle=False, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=CONFIG['data']['batch_size'], shuffle=False, num_workers=0)
    
    # Model, loss, optimizer
    seed_everything(CONFIG['seed'] + fold)
    model = DenseNetHeadModel(
        cnn_dim=CONFIG['model']['cnn_emb_dim'],
        n_pri=CONFIG['model']['primary_num_classes'],
        n_sub=CONFIG['model']['subtype_num_classes'],
        dropout=CONFIG['model']['dropout']
    ).to(device)
    
    crit_p = nn.CrossEntropyLoss(weight=pri_weights_tensor, label_smoothing=CONFIG['training']['label_smoothing'])
    crit_s = nn.CrossEntropyLoss(weight=sub_weights_tensor, label_smoothing=CONFIG['training']['label_smoothing'])
    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['training']['lr_head'], weight_decay=CONFIG['training']['weight_decay'])
    scaler = torch.amp.GradScaler('cuda', enabled=CONFIG['training']['mixed_precision'])
    early_stopping = EarlyStopping(patience=CONFIG['training']['early_stopping_patience'], mode='max')
    
    for epoch in range(CONFIG['training']['phase1_epochs']):
        tm = train_cached_epoch(model, train_loader, optimizer, crit_p, crit_s, scaler)
        vm, _ = eval_cached_epoch(model, val_loader, crit_p, crit_s)
        improved = early_stopping(vm['subtype_macro_f1'], model)
        if early_stopping.early_stop:
            break
            
    # Load best inner model state and evaluate on strictly held-out outer test fold
    model.load_state_dict(early_stopping.best_state)
    test_m, (tp, pp, ts, ps) = eval_cached_epoch(model, test_loader, crit_p, crit_s)
    
    b1_fold_results.append(test_m)
    b1_per_class_f1_list.append(test_m['subtype_per_class_f1'])
    print(f"   Fold {fold + 1} Test Result -> Subtype Acc: {test_m['subtype_acc']:.4f} | Subtype Macro-F1: {test_m['subtype_macro_f1']:.4f} | Primary Acc: {test_m['primary_acc']:.4f}")

b1_sub_f1_mean = np.mean([r['subtype_macro_f1'] for r in b1_fold_results])
b1_sub_f1_std = np.std([r['subtype_macro_f1'] for r in b1_fold_results])
print(f"\n[OK] Baseline 1 (DenseNet-201) 5-Fold Subtype Macro-F1: {b1_sub_f1_mean:.4f} +/- {b1_sub_f1_std:.4f}")

## Task B: Baseline 2 (ViT-B/16) Retraining & 5-Fold Cross-Validation

Retrain ViT-B/16 multi-task heads under the corrected 5-fold CV protocol with nested patient-level inner validation for model selection.

In [ ]:
# ============================================================
# Baseline 2 (ViT-B/16) 5-Fold Cross-Validation
# ============================================================
print("=" * 60)
print("BASELINE 2 (ViT-B/16): 5-FOLD CROSS-VALIDATION")
print("=" * 60)

# Linear head model on top of cached ViT features
class ViTHeadModel(nn.Module):
    def __init__(self, vit_dim=768, n_pri=2, n_sub=8, dropout=0.3):
        super().__init__()
        self.head_primary = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(vit_dim, n_pri)
        )
        self.head_subtype = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(vit_dim, n_sub)
        )
    def forward_from_embeddings(self, cnn_emb=None, vit_emb=None):
        return self.head_primary(vit_emb), self.head_subtype(vit_emb), vit_emb

b2_fold_results = []
b2_per_class_f1_list = []

for fold in range(CONFIG['training']['k_folds']):
    print(f"\n>>> Fold {fold + 1}/{CONFIG['training']['k_folds']}")
    
    # Outer split
    outer_test_mask = (all_folds == fold)
    outer_train_mask = (all_folds != fold)
    
    # Nested patient-level inner split
    outer_train_df = folds_task_b_df[outer_train_mask].copy()
    inner_val_pids = []
    for st, grp in outer_train_df.groupby('subtype_label'):
        pids = list(grp['patient_id'].unique())
        random.seed(CONFIG['seed'] + fold * 10)
        random.shuffle(pids)
        n = len(pids)
        n_val = max(1, int(round(n * 0.2))) if n >= 4 else (1 if n >= 2 else 0)
        inner_val_pids.extend(pids[:n_val])
        
    inner_val_mask = outer_train_mask & folds_task_b_df['patient_id'].isin(inner_val_pids).values
    inner_train_mask = outer_train_mask & (~folds_task_b_df['patient_id'].isin(inner_val_pids).values)
    
    # DataLoaders (with drop_last=True for training to avoid batch size 1 BatchNorm crashes)
    train_ds = CachedFeatureDataset(
        all_cnn_feats[inner_train_mask], all_vit_feats[inner_train_mask],
        all_primary_targets[inner_train_mask], all_subtype_targets[inner_train_mask]
    )
    val_ds = CachedFeatureDataset(
        all_cnn_feats[inner_val_mask], all_vit_feats[inner_val_mask],
        all_primary_targets[inner_val_mask], all_subtype_targets[inner_val_mask]
    )
    test_ds = CachedFeatureDataset(
        all_cnn_feats[outer_test_mask], all_vit_feats[outer_test_mask],
        all_primary_targets[outer_test_mask], all_subtype_targets[outer_test_mask]
    )
    
    train_loader = DataLoader(train_ds, batch_size=CONFIG['data']['batch_size'], shuffle=True, num_workers=0, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=CONFIG['data']['batch_size'], shuffle=False, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=CONFIG['data']['batch_size'], shuffle=False, num_workers=0)
    
    # Model, loss, optimizer
    seed_everything(CONFIG['seed'] + fold)
    model = ViTHeadModel(
        vit_dim=CONFIG['model']['vit_emb_dim'],
        n_pri=CONFIG['model']['primary_num_classes'],
        n_sub=CONFIG['model']['subtype_num_classes'],
        dropout=CONFIG['model']['dropout']
    ).to(device)
    
    crit_p = nn.CrossEntropyLoss(weight=pri_weights_tensor, label_smoothing=CONFIG['training']['label_smoothing'])
    crit_s = nn.CrossEntropyLoss(weight=sub_weights_tensor, label_smoothing=CONFIG['training']['label_smoothing'])
    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['training']['lr_head'], weight_decay=CONFIG['training']['weight_decay'])
    scaler = torch.amp.GradScaler('cuda', enabled=CONFIG['training']['mixed_precision'])
    early_stopping = EarlyStopping(patience=CONFIG['training']['early_stopping_patience'], mode='max')
    
    for epoch in range(CONFIG['training']['phase1_epochs']):
        tm = train_cached_epoch(model, train_loader, optimizer, crit_p, crit_s, scaler)
        vm, _ = eval_cached_epoch(model, val_loader, crit_p, crit_s)
        improved = early_stopping(vm['subtype_macro_f1'], model)
        if early_stopping.early_stop:
            break
            
    # Load best inner model state and evaluate on strictly held-out outer test fold
    model.load_state_dict(early_stopping.best_state)
    test_m, (tp, pp, ts, ps) = eval_cached_epoch(model, test_loader, crit_p, crit_s)
    
    b2_fold_results.append(test_m)
    b2_per_class_f1_list.append(test_m['subtype_per_class_f1'])
    print(f"   Fold {fold + 1} Test Result -> Subtype Acc: {test_m['subtype_acc']:.4f} | Subtype Macro-F1: {test_m['subtype_macro_f1']:.4f} | Primary Acc: {test_m['primary_acc']:.4f}")

b2_sub_f1_mean = np.mean([r['subtype_macro_f1'] for r in b2_fold_results])
b2_sub_f1_std = np.std([r['subtype_macro_f1'] for r in b2_fold_results])
print(f"\n[OK] Baseline 2 (ViT-B/16) 5-Fold Subtype Macro-F1: {b2_sub_f1_mean:.4f} +/- {b2_sub_f1_std:.4f}")

## Task B: Baseline 3 (DenseNet + ViT Fusion) 5-Fold Cross-Validation (Phase 1)

Evaluate the feature-level fusion hybrid model ($1920 \to 384 + 768 \to 384 \to 768 \to 384 \to \text{heads}$) across all 5 folds using the exact same nested inner validation protocol.

In [ ]:
# ============================================================
# Baseline 3 (Feature Fusion Hybrid) 5-Fold Cross-Validation
# ============================================================
print("=" * 60)
print("BASELINE 3 (DenseNet-201 + ViT-B/16 Fusion): 5-FOLD CROSS-VALIDATION")
print("=" * 60)

# Projection & Fusion Module for cached embeddings
class FeatureFusionHeadOnly(nn.Module):
    def __init__(self, config):
        super().__init__()
        cnn_dim = config['model']['cnn_emb_dim']
        vit_dim = config['model']['vit_emb_dim']
        proj_dim = config['model']['proj_dim']
        mlp_hidden = config['model']['fusion_hidden_dim']
        dropout = config['model']['dropout']
        n_pri = config['model']['primary_num_classes']
        n_sub = config['model']['subtype_num_classes']

        self.cnn_proj = nn.Sequential(
            nn.Linear(cnn_dim, proj_dim),
            nn.BatchNorm1d(proj_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout)
        )
        self.vit_proj = nn.Sequential(
            nn.Linear(vit_dim, proj_dim),
            nn.LayerNorm(proj_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout)
        )
        self.fusion_mlp = nn.Sequential(
            nn.Linear(proj_dim * 2, mlp_hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout)
        )
        self.head_primary = nn.Linear(mlp_hidden, n_pri)
        self.head_subtype = nn.Linear(mlp_hidden, n_sub)

    def forward_from_embeddings(self, cnn_emb, vit_emb):
        p_cnn = self.cnn_proj(cnn_emb)
        p_vit = self.vit_proj(vit_emb)
        fused = torch.cat([p_cnn, p_vit], dim=1)
        hidden = self.fusion_mlp(fused)
        return self.head_primary(hidden), self.head_subtype(hidden), hidden

b3_fold_results = []
b3_per_class_f1_list = []
all_b3_test_preds = {'pri_t': [], 'pri_p': [], 'sub_t': [], 'sub_p': []}

for fold in range(CONFIG['training']['k_folds']):
    print(f"\n>>> Fold {fold + 1}/{CONFIG['training']['k_folds']}")
    
    # Outer split
    outer_test_mask = (all_folds == fold)
    outer_train_mask = (all_folds != fold)
    
    # Nested patient-level inner split
    outer_train_df = folds_task_b_df[outer_train_mask].copy()
    inner_val_pids = []
    for st, grp in outer_train_df.groupby('subtype_label'):
        pids = list(grp['patient_id'].unique())
        random.seed(CONFIG['seed'] + fold * 10)
        random.shuffle(pids)
        n = len(pids)
        n_val = max(1, int(round(n * 0.2))) if n >= 4 else (1 if n >= 2 else 0)
        inner_val_pids.extend(pids[:n_val])
        
    inner_val_mask = outer_train_mask & folds_task_b_df['patient_id'].isin(inner_val_pids).values
    inner_train_mask = outer_train_mask & (~folds_task_b_df['patient_id'].isin(inner_val_pids).values)
    
    # DataLoaders (with drop_last=True for training to avoid batch size 1 BatchNorm crashes)
    train_ds = CachedFeatureDataset(
        all_cnn_feats[inner_train_mask], all_vit_feats[inner_train_mask],
        all_primary_targets[inner_train_mask], all_subtype_targets[inner_train_mask]
    )
    val_ds = CachedFeatureDataset(
        all_cnn_feats[inner_val_mask], all_vit_feats[inner_val_mask],
        all_primary_targets[inner_val_mask], all_subtype_targets[inner_val_mask]
    )
    test_ds = CachedFeatureDataset(
        all_cnn_feats[outer_test_mask], all_vit_feats[outer_test_mask],
        all_primary_targets[outer_test_mask], all_subtype_targets[outer_test_mask]
    )
    
    train_loader = DataLoader(train_ds, batch_size=CONFIG['data']['batch_size'], shuffle=True, num_workers=0, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=CONFIG['data']['batch_size'], shuffle=False, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=CONFIG['data']['batch_size'], shuffle=False, num_workers=0)
    
    # Model, loss, optimizer
    seed_everything(CONFIG['seed'] + fold)
    model = FeatureFusionHeadOnly(CONFIG).to(device)
    
    crit_p = nn.CrossEntropyLoss(weight=pri_weights_tensor, label_smoothing=CONFIG['training']['label_smoothing'])
    crit_s = nn.CrossEntropyLoss(weight=sub_weights_tensor, label_smoothing=CONFIG['training']['label_smoothing'])
    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['training']['lr_head'], weight_decay=CONFIG['training']['weight_decay'])
    scaler = torch.amp.GradScaler('cuda', enabled=CONFIG['training']['mixed_precision'])
    early_stopping = EarlyStopping(patience=CONFIG['training']['early_stopping_patience'], mode='max')
    
    for epoch in range(CONFIG['training']['phase1_epochs']):
        tm = train_cached_epoch(model, train_loader, optimizer, crit_p, crit_s, scaler)
        vm, _ = eval_cached_epoch(model, val_loader, crit_p, crit_s)
        improved = early_stopping(vm['subtype_macro_f1'], model)
        if early_stopping.early_stop:
            break
            
    # Load best inner model state and evaluate on strictly held-out outer test fold
    model.load_state_dict(early_stopping.best_state)
    test_m, (tp, pp, ts, ps) = eval_cached_epoch(model, test_loader, crit_p, crit_s)
    
    b3_fold_results.append(test_m)
    b3_per_class_f1_list.append(test_m['subtype_per_class_f1'])
    
    all_b3_test_preds['pri_t'].extend(tp)
    all_b3_test_preds['pri_p'].extend(pp)
    all_b3_test_preds['sub_t'].extend(ts)
    all_b3_test_preds['sub_p'].extend(ps)
    
    print(f"   Fold {fold + 1} Test Result -> Subtype Acc: {test_m['subtype_acc']:.4f} | Subtype Macro-F1: {test_m['subtype_macro_f1']:.4f} | Primary Acc: {test_m['primary_acc']:.4f}")

b3_sub_f1_mean = np.mean([r['subtype_macro_f1'] for r in b3_fold_results])
b3_sub_f1_std = np.std([r['subtype_macro_f1'] for r in b3_fold_results])
print(f"\n[OK] Baseline 3 (Fusion) 5-Fold Subtype Macro-F1: {b3_sub_f1_mean:.4f} +/- {b3_sub_f1_std:.4f}")

## Phase 2: End-to-End Fine-Tuning with Partial Unfreezing

Unfreeze DenseNet-201 `denseblock4` + `norm5` and ViT-B/16 final 2 transformer blocks + `norm`. Train end-to-end with discriminative learning rates (`1e-5` for backbones, `1e-3` for fusion heads).

In [ ]:
# ============================================================
# Phase 2: End-to-End Fine-Tuning with Partial Unfreeze
# ============================================================
print("=" * 60)
print("PHASE 2: END-TO-END FINE-TUNING")
print("=" * 60)

# Build Task A DataLoaders
train_a_df = split_task_a_df[split_task_a_df['split'] == 'train'].copy()
val_a_df   = split_task_a_df[split_task_a_df['split'] == 'val'].copy()
test_a_df  = split_task_a_df[split_task_a_df['split'] == 'test'].copy()

train_a_ds = BreakHisDataset(train_a_df, transform=train_transform)
val_a_ds   = BreakHisDataset(val_a_df, transform=val_transform)
test_a_ds  = BreakHisDataset(test_a_df, transform=val_transform)

# Set drop_last=True for training to avoid batch size 1 BatchNorm crashes
train_a_loader = DataLoader(train_a_ds, batch_size=CONFIG['data']['batch_size'], shuffle=True, num_workers=0, pin_memory=True, drop_last=True)
val_a_loader   = DataLoader(val_a_ds, batch_size=CONFIG['data']['batch_size'], shuffle=False, num_workers=0, pin_memory=True)
test_a_loader  = DataLoader(test_a_ds, batch_size=CONFIG['data']['batch_size'], shuffle=False, num_workers=0, pin_memory=True)

# Build full model and unfreeze partial blocks
seed_everything(CONFIG['seed'])
full_hybrid_model = FeatureFusionMultiTaskModel(CONFIG).to(device)
full_hybrid_model.freeze_backbones()
full_hybrid_model.unfreeze_partial_backbones(vit_blocks=CONFIG['training']['unfreeze_vit_last_n_blocks'])

# Discriminative parameters
backbone_params = []
head_params = []
for name, param in full_hybrid_model.named_parameters():
    if param.requires_grad:
        if 'cnn_features' in name or 'vit_backbone' in name:
            backbone_params.append(param)
        else:
            head_params.append(param)

optimizer_p2 = torch.optim.AdamW([
    {'params': backbone_params, 'lr': CONFIG['training']['lr_backbone']},
    {'params': head_params, 'lr': CONFIG['training']['lr_head']}
], weight_decay=CONFIG['training']['weight_decay'])

crit_p = nn.CrossEntropyLoss(weight=pri_weights_tensor, label_smoothing=CONFIG['training']['label_smoothing'])
crit_s = nn.CrossEntropyLoss(weight=sub_weights_tensor, label_smoothing=CONFIG['training']['label_smoothing'])
scaler_p2 = torch.amp.GradScaler('cuda', enabled=CONFIG['training']['mixed_precision'])
early_stopping_p2 = EarlyStopping(patience=CONFIG['training']['early_stopping_patience'], mode='max')

print(f"[OK] Training Phase 2 for up to {CONFIG['training']['phase2_epochs']} epochs...")

for epoch in range(CONFIG['training']['phase2_epochs']):
    full_hybrid_model.train()
    total_loss, n = 0.0, 0
    
    for imgs, yp, ys in train_a_loader:
        imgs = imgs.to(device, non_blocking=True)
        yp = yp.to(device, non_blocking=True)
        ys = ys.to(device, non_blocking=True)

        optimizer_p2.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=CONFIG['training']['mixed_precision']):
            logits_p, logits_s, _ = full_hybrid_model(imgs)
            loss = crit_p(logits_p, yp) + CONFIG['training']['subtype_loss_weight'] * crit_s(logits_s, ys)

        scaler_p2.scale(loss).backward()
        scaler_p2.step(optimizer_p2)
        scaler_p2.update()
        total_loss += loss.item() * yp.size(0)
        n += yp.size(0)
        
    train_loss = total_loss / max(n, 1)

    # Validation
    full_hybrid_model.eval()
    val_loss, vn = 0.0, 0
    vp_p, vt_p, vp_s, vt_s = [], [], [], []
    with torch.no_grad():
        for imgs, yp, ys in val_a_loader:
            imgs = imgs.to(device, non_blocking=True)
            yp = yp.to(device, non_blocking=True)
            ys = ys.to(device, non_blocking=True)
            with torch.amp.autocast('cuda', enabled=CONFIG['training']['mixed_precision']):
                logits_p, logits_s, _ = full_hybrid_model(imgs)
                loss = crit_p(logits_p, yp) + CONFIG['training']['subtype_loss_weight'] * crit_s(logits_s, ys)
            val_loss += loss.item() * yp.size(0)
            vn += yp.size(0)
            vp_p.append(logits_p.argmax(1).cpu()); vt_p.append(yp.cpu())
            vp_s.append(logits_s.argmax(1).cpu()); vt_s.append(ys.cpu())

    vm = compute_metrics(
        torch.cat(vt_p).numpy(), torch.cat(vp_p).numpy(),
        torch.cat(vt_s).numpy(), torch.cat(vp_s).numpy()
    )
    
    improved = early_stopping_p2(vm['subtype_macro_f1'], full_hybrid_model)
    star = " [BEST]" if improved else ""
    print(f"Ep {epoch+1:02d}/{CONFIG['training']['phase2_epochs']} | Train Loss: {train_loss:.4f} | Val Sub-F1: {vm['subtype_macro_f1']:.4f} | Val Pri-F1: {vm['primary_macro_f1']:.4f}{star}")
    
    if early_stopping_p2.early_stop:
        print(f"[INFO] Early stopping triggered at epoch {epoch+1}.")
        break

# Save best model checkpoint
best_ckpt_path = CONFIG['paths']['best_fusion_checkpoint']
torch.save({
    'model_state_dict': early_stopping_p2.best_state,
    'config': CONFIG,
    'best_val_subtype_f1': early_stopping_p2.best_score
}, best_ckpt_path)
print(f"[OK] Best fusion model checkpoint saved to: {best_ckpt_path}")

## Task A: Primary Classification Evaluation on Held-Out Test Set

Evaluate the best hybrid model on the single held-out test partition (`split_task_a.csv`).

In [ ]:
# ============================================================
# Final Test Evaluation on Task A Held-Out Partition
# ============================================================
print("=" * 60)
print("TASK A HELD-OUT TEST EVALUATION")
print("=" * 60)

# Load best checkpoint
full_hybrid_model.load_state_dict(torch.load(CONFIG['paths']['best_fusion_checkpoint'], weights_only=False)['model_state_dict'])
full_hybrid_model.eval()

tp_p, tt_p, tp_s, tt_s = [], [], [], []
with torch.no_grad():
    for imgs, yp, ys in test_a_loader:
        imgs = imgs.to(device, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=CONFIG['training']['mixed_precision']):
            lp, ls, _ = full_hybrid_model(imgs)
        tp_p.append(lp.argmax(1).cpu()); tt_p.append(yp)
        tp_s.append(ls.argmax(1).cpu()); tt_s.append(ys)

tp_p = torch.cat(tp_p).numpy(); tt_p = torch.cat(tt_p).numpy()
tp_s = torch.cat(tp_s).numpy(); tt_s = torch.cat(tt_s).numpy()

pri_acc = accuracy_score(tt_p, tp_p)
pri_prec = precision_score(tt_p, tp_p, average='macro', zero_division=0)
pri_rec = recall_score(tt_p, tp_p, average='macro', zero_division=0)
pri_f1 = f1_score(tt_p, tp_p, average='macro', zero_division=0)

print("Task A (Primary Classification) Held-Out Results:")
print(f"  Accuracy:  {pri_acc:.4f}")
print(f"  Precision: {pri_prec:.4f}")
print(f"  Recall:    {pri_rec:.4f}")
print(f"  Macro-F1:  {pri_f1:.4f}")
print("\n" + classification_report(tt_p, tp_p, target_names=[PRIMARY_IDX_TO_LABEL[i] for i in range(2)], zero_division=0))

## Comprehensive 3-Way Scientific Comparison & Pre-Registered Success Criterion

### Pre-Registered Success Criterion Rule:
$$\text{Fusion Subtype Macro-F1} \ge \max(\text{DenseNet Macro-F1}, \text{ViT Macro-F1}) + 1.0 \times \sigma_{\text{cross-fold}}$$

We compare Baseline 1 (DenseNet-201), Baseline 2 (ViT-B/16), and Baseline 3 (Fusion) evaluated under the exact same 5-fold cross-validation protocol.

In [ ]:
# ============================================================
# 3-Way Benchmark Comparison & Pre-Registered Verdict
# ============================================================
print("=" * 60)
print("BENCHMARK COMPARISON: BASELINE 1 vs BASELINE 2 vs BASELINE 3")
print("=" * 60)

# Compute metrics summary
best_baseline_mean = max(b1_sub_f1_mean, b2_sub_f1_mean)
baseline_std = b1_sub_f1_std if best_baseline_mean == b1_sub_f1_mean else b2_sub_f1_std
threshold_target = best_baseline_mean + 1.0 * baseline_std
meets_threshold = b3_sub_f1_mean >= threshold_target

comparison_df = pd.DataFrame([
    {
        'Model': 'Baseline 1: DenseNet-201',
        'Architecture Type': 'CNN',
        'Subtype Macro-F1 (Mean +/- Std)': f"{b1_sub_f1_mean:.4f} +/- {b1_sub_f1_std:.4f}",
        'Subtype Accuracy (Mean)': f"{np.mean([r['subtype_acc'] for r in b1_fold_results]):.4f}",
        'Primary Accuracy (Mean)': f"{np.mean([r['primary_acc'] for r in b1_fold_results]):.4f}",
    },
    {
        'Model': 'Baseline 2: ViT-B/16',
        'Architecture Type': 'Vision Transformer',
        'Subtype Macro-F1 (Mean +/- Std)': f"{b2_sub_f1_mean:.4f} +/- {b2_sub_f1_std:.4f}",
        'Subtype Accuracy (Mean)': f"{np.mean([r['subtype_acc'] for r in b2_fold_results]):.4f}",
        'Primary Accuracy (Mean)': f"{np.mean([r['primary_acc'] for r in b2_fold_results]):.4f}",
    },
    {
        'Model': 'Baseline 3: DenseNet + ViT Fusion',
        'Architecture Type': 'Feature-Level Hybrid',
        'Subtype Macro-F1 (Mean +/- Std)': f"{b3_sub_f1_mean:.4f} +/- {b3_sub_f1_std:.4f}",
        'Subtype Accuracy (Mean)': f"{np.mean([r['subtype_acc'] for r in b3_fold_results]):.4f}",
        'Primary Accuracy (Mean)': f"{np.mean([r['primary_acc'] for r in b3_fold_results]):.4f}",
    }
])

print("\n--- Summary Benchmark Table ---")
print(comparison_df.to_string(index=False))

# Pre-registered success test
print("\n" + "=" * 60)
print("PRE-REGISTERED SUCCESS CRITERION VERDICT")
print("=" * 60)
print(f"  Best Single Baseline Macro-F1:     {best_baseline_mean:.4f}")
print(f"  Cross-Fold Standard Deviation:    {baseline_std:.4f}")
print(f"  Pre-Registered Success Target:    {threshold_target:.4f}")
print(f"  Baseline 3 Fusion Macro-F1:       {b3_sub_f1_mean:.4f}")

if meets_threshold:
    verdict = "[VERDICT: SUCCESS] Feature-level fusion successfully exceeded the pre-registered threshold!"
else:
    verdict = "[VERDICT: INCONCLUSIVE / NEGATIVE] Simple concatenation fusion did not exceed the 1.0-sigma threshold above the stronger baseline."
print(f"\n{verdict}")

# Save comparison CSV
comp_path = CONFIG['paths']['comparison_csv']
comparison_df.to_csv(comp_path, index=False)
print(f"\n[OK] Comparison table saved to: {comp_path}")

In [ ]:
# ============================================================
# Visualizations: Confusion Matrices & Per-Class F1 Analysis
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# 1. Primary Classification Confusion Matrix
pri_names = [PRIMARY_IDX_TO_LABEL[i] for i in range(2)]
pri_cm = confusion_matrix(tt_p, tp_p)
sns.heatmap(pri_cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=pri_names, yticklabels=pri_names)
axes[0].set_title('Task A: Primary Confusion Matrix (Held-Out Test)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('True Label')

# 2. Subtype 5-Fold Aggregated Confusion Matrix
sub_names = [SUBTYPE_IDX_TO_LABEL[i] for i in range(8)]
sub_cm = confusion_matrix(all_b3_test_preds['sub_t'], all_b3_test_preds['sub_p'])
sns.heatmap(sub_cm, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=[s[:6] for s in sub_names], yticklabels=sub_names)
axes[1].set_title('Task B: Subtype 5-Fold Aggregated Confusion Matrix', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Predicted Subtype')
axes[1].set_ylabel('True Subtype')

plt.tight_layout()
cm_plot_path = os.path.join(CONFIG['paths']['save_dir'], 'confusion_matrices_base3.png')
plt.savefig(cm_plot_path, dpi=300)
plt.show()
print(f"[OK] Confusion matrix figure saved to: {cm_plot_path}")

# 3. Per-Class Subtype F1 Comparison Bar Chart
fig, ax = plt.subplots(figsize=(12, 6))
b1_pc_means = np.mean(np.array(b1_per_class_f1_list), axis=0)
b2_pc_means = np.mean(np.array(b2_per_class_f1_list), axis=0)
b3_pc_means = np.mean(np.array(b3_per_class_f1_list), axis=0)

x = np.arange(len(sub_names))
w = 0.25
ax.bar(x - w, b1_pc_means, width=w, label='DenseNet-201 (CNN)', color='#4A90E2')
ax.bar(x, b2_pc_means, width=w, label='ViT-B/16 (Transformer)', color='#50E3C2')
ax.bar(x + w, b3_pc_means, width=w, label='Baseline 3 (Fusion)', color='#F5A623')

ax.set_xticks(x)
ax.set_xticklabels(sub_names, rotation=35, ha='right')
ax.set_ylabel('Subtype Macro F1-Score')
ax.set_title('Per-Class Subtype F1 Comparison across Models (5-Fold Mean)', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
f1_plot_path = os.path.join(CONFIG['paths']['save_dir'], 'per_class_f1_comparison.png')
plt.savefig(f1_plot_path, dpi=300)
plt.show()
print(f"[OK] Per-class F1 bar chart saved to: {f1_plot_path}")

## Deliverables Checklist & Summary

| # | Artifact Description | Path / Destination | Status |
|---|---|---|---|
| 1 | Canonical Task A 3-Way Split | `/content/drive/MyDrive/output_base3/split_task_a.csv` | [DONE] Verified & Saved |
| 2 | Canonical Task B 5-Fold Assignments | `/content/drive/MyDrive/output_base3/folds_task_b.csv` | [DONE] Verified & Saved |
| 3 | Baseline 1 (DenseNet-201) 5-Fold Retraining | In-memory evaluation & logging | [DONE] Completed |
| 4 | Baseline 2 (ViT-B/16) 5-Fold Retraining | In-memory evaluation & logging | [DONE] Completed |
| 5 | Baseline 3 (Fusion) 5-Fold Cross-Validation | In-memory evaluation & logging | [DONE] Completed |
| 6 | Best Fine-Tuned Hybrid Checkpoint | `/content/drive/MyDrive/output_base3/best_fusion_model.pth` | [DONE] Saved to Drive |
| 7 | 3-Way Benchmark Comparison Table | `/content/drive/MyDrive/output_base3/benchmark_comparison.csv` | [DONE] Saved to Drive |
| 8 | Confusion Matrices & Diagnostic Visualizations | `/content/drive/MyDrive/output_base3/confusion_matrices_base3.png` | [DONE] Saved to Drive |
| 9 | Per-Class F1 Analysis Plot | `/content/drive/MyDrive/output_base3/per_class_f1_comparison.png` | [DONE] Saved to Drive |
| 10 | Pre-Registered Hypothesis Verdict | Documented in Cell 29 | [DONE] Verified |